<a href="https://colab.research.google.com/github/chiptune234u-lgtm/SandProject/blob/main/test_sand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#@title ⏳ サンドボックス・エディター（高速化版）

#@markdown ### 1. サイクル設定
CYCLE_DURATION = "30s" #@param ["20s", "30s"]

# ここでDURATIONを確定
if CYCLE_DURATION == "30s":
    DURATION = 30
else:
    DURATION = 20

#@markdown ### 2. 基本設定
MODE = "sand"
FPS = 30 # 高速化したのでFPSを少し上げても大丈夫です
BLOCK_SIZE = 12

#@markdown ### 3. 演出設定
PREFILL_HEIGHT = 0.4

#@markdown ### 4. 風設定
WIND_STRENGTH = 0.3

import numpy as np
import cv2
import os
import glob
import random
from google.colab import drive
import colorsys
from numba import jit

# ドライブマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

WIDTH, HEIGHT = 720, 1280
COLS, ROWS = WIDTH // BLOCK_SIZE, HEIGHT // BLOCK_SIZE

# --- 動的な色生成ロジック ---\
BASE_HUE = random.random()

def get_dynamic_color():
    h = (BASE_HUE + random.uniform(-0.05, 0.05)) % 1.0
    s = random.uniform(0.5, 0.8)
    v = random.uniform(0.8, 1.0)
    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return [int(r*255), int(g*255), int(b*255)]

# 背景準備
bg_dir = "/content/drive/MyDrive/Colab Notebooks/画像素材"
bg_files = glob.glob(os.path.join(bg_dir, "*"))
if bg_files:
    bg_img = cv2.imread(random.choice(bg_files))
    bg_img = cv2.resize(bg_img, (WIDTH, HEIGHT))
    bg_img = cv2.cvtColor(bg_img, cv2.COLOR_BGR2RGB)
else:
    bg_img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

# --- グリッド初期化（天井に砂をセット） ---\
grid = np.zeros((ROWS, COLS, 3), dtype=np.uint8)
fill_limit = int(ROWS * PREFILL_HEIGHT)

for y in range(0, fill_limit):
    width_at_y = int((fill_limit - y) * 1.5)
    center = COLS // 2
    # NumPyのスライス代入で高速化
    start_x = max(0, center - width_at_y)
    end_x = min(COLS, center + width_at_y)
    if end_x > start_x:
        # ランダムなマスク生成
        mask = np.random.random(end_x - start_x) > 0.1
        # 色を生成して適用（少し簡易化して高速化）
        colors = np.array([get_dynamic_color() for _ in range(np.sum(mask))])
        if len(colors) > 0:
            grid[y, start_x:end_x][mask] = colors

print(f"初期化完了: モード={DURATION}s, 天井砂セット完了！")

初期化完了: モード=30s, 天井砂セット完了！


In [ ]:
velocity_grid = np.zeros((ROWS, COLS, 2), dtype=np.float32)

# 爆発計算をベクトル化（Pythonループを排除して高速化）
def apply_deep_explosion(current_grid, v_grid, power=30):
    # 砂がある座標を取得
    active_mask = np.any(current_grid > 0, axis=2)
    active_y, active_x = np.where(active_mask)

    if len(active_y) == 0:
        return v_grid, (COLS//2, ROWS//2)

    min_x, max_x = np.min(active_x), np.max(active_x)
    explosion_x = random.randint(min_x, max_x)

    min_y, max_y = np.min(active_y), np.max(active_y)
    explosion_y = int(max_y - (max_y - min_y) * 0.2)

    # メッシュグリッドで距離計算を一括処理
    y_indices, x_indices = np.indices((ROWS, COLS))
    dy = y_indices - explosion_y
    dx = x_indices - explosion_x
    dist = np.sqrt(dx**2 + dy**2) + 0.1 # 0除算防止

    # 爆発範囲内のマスク作成
    explosion_mask = (dist < 60) & active_mask

    if np.any(explosion_mask):
        force = power / dist[explosion_mask]
        v_grid[explosion_mask, 0] += (dx[explosion_mask] / dist[explosion_mask]) * force * 15
        v_grid[explosion_mask, 1] += (dy[explosion_mask] / dist[explosion_mask]) * force * 15

    return v_grid, (explosion_x, explosion_y)

# Numbaで風計算を高速化
@jit(nopython=True)
def apply_wind_numba(v_grid, sand_mask, wind_force):
    # sand_maskは (ROWS, COLS) のbool配列
    rows, cols = v_grid.shape[:2]
    for y in range(rows - 1): # 最下段は除外
        for x in range(cols):
            if sand_mask[y, x]:
                v_grid[y, x, 0] += wind_force
    return v_grid

def apply_wind(current_grid, v_grid, wind_force):
    sand_mask = np.any(current_grid > 0, axis=2)
    return apply_wind_numba(v_grid, sand_mask, wind_force)

# 物理更新ロジック（既存のベクトル化を維持しつつ整理）
def update_physics_vectorized(current_grid, v_grid, mode):
    new_grid = np.zeros_like(current_grid)
    new_v_grid = np.zeros_like(v_grid)

    # 下から上へスキャン
    for y in range(ROWS - 1, -1, -1):
        # その行に砂があるかチェック（高速化のための早期リターン）
        if not np.any(current_grid[y]): continue

        sand_mask_in_row = np.any(current_grid[y, :, :], axis=1)
        sand_x_coords = np.where(sand_mask_in_row)[0]

        colors = current_grid[y, sand_x_coords]
        velocities = v_grid[y, sand_x_coords]

        vx, vy = velocities[:, 0], velocities[:, 1]

        # 移動先計算
        proposed_nx = np.clip((sand_x_coords + vx).astype(int), 0, COLS - 1)
        proposed_ny = (y + vy + 1).astype(int)

        new_vx, new_vy = vx * 0.9, vy * 0.9 # 減衰

        # --- 境界チェック ---
        falling_off_mask = proposed_ny >= ROWS

        # 画面外へ落ちる処理
        if np.any(falling_off_mask):
            target_x_fall = proposed_nx[falling_off_mask]
            # 最下段に固定
            new_grid[ROWS-1, target_x_fall] = colors[falling_off_mask]
            new_v_grid[ROWS-1, target_x_fall] = 0.0

        # 画面内での移動処理
        remaining_mask = ~falling_off_mask
        if not np.any(remaining_mask): continue

        sand_x_rem = sand_x_coords[remaining_mask]
        colors_rem = colors[remaining_mask]
        p_nx = proposed_nx[remaining_mask]
        p_ny = proposed_ny[remaining_mask]
        n_vx, n_vy = new_vx[remaining_mask], new_vy[remaining_mask]

        # Ensure p_ny is within valid row bounds
        p_ny = np.clip(p_ny, 0, ROWS - 1)

        # 衝突判定（同じ場所に行こうとする砂の競合解決）
        target_indices_linear = p_ny * COLS + p_nx
        unique_indices, first_indices = np.unique(target_indices_linear, return_index=True)

        # 移動先が空いているか確認
        # unique_indices から y, x を復元
        u_y, u_x = np.divmod(unique_indices, COLS)
        can_move = np.all(new_grid[u_y, u_x] == 0, axis=1)

        valid_indices = first_indices[can_move]

        # 移動確定
        if len(valid_indices) > 0:
            final_y = p_ny[valid_indices]
            final_x = p_nx[valid_indices]
            new_grid[final_y, final_x] = colors_rem[valid_indices]
            new_v_grid[final_y, final_x, 0] = n_vx[valid_indices]
            new_v_grid[final_y, final_x, 1] = n_vy[valid_indices]

        # 移動できなかった砂（スライド処理へ）
        # 元のインデックス配列のうち、valid_indicesに含まれないものが移動失敗
        # ここは簡易化のために元の位置に残すか、ランダムスライド
        mask_moved = np.zeros(len(sand_x_rem), dtype=bool)
        mask_moved[valid_indices] = True
        stuck_indices = np.where(~mask_moved)[0]

        if len(stuck_indices) > 0:
             for i in stuck_indices:
                curr_x = sand_x_rem[i]
                curr_c = colors_rem[i]

                # 左右どちらかに滑る
                slide = random.choice([-1, 1])
                tx = curr_x + slide
                ty = y + 1

                if 0 <= tx < COLS and ty < ROWS and np.all(new_grid[ty, tx] == 0):
                    new_grid[ty, tx] = curr_c
                    new_v_grid[ty, tx] = [n_vx[i] * 0.5, 0]
                else:
                    new_grid[y, curr_x] = curr_c
                    new_v_grid[y, curr_x] = [0, 0]

    return new_grid, new_v_grid

In [ ]:

import time
import math

# 第1セルのDURATIONを元に全フレーム数を確定
total_frames = FPS * DURATION
frames_out = []

# --- タイミングの比率計算 ---\
ratio = DURATION / 20.0
EXPLOSION_FRAMES = [0, int(FPS * 15 * ratio)]
EMIT_STOP_F = int(FPS * 17 * ratio)

# 回転演出
ROT_SEC = 2.5
ROTATION_START_F = total_frames - int(FPS * ROT_SEC)

# 風のパラメータ（事前計算テーブル作成で高速化）
WIND_MAX_STRENGTH = WIND_STRENGTH
WIND_START_F = FPS * 7
WIND_PEAK_F  = FPS * 10
WIND_END_F   = FPS * 13
WIND_DIRECTION = random.choice([-1, 1])

# 風の力を配列で事前に計算
wind_forces = np.zeros(total_frames)
t_in = np.arange(WIND_START_F, WIND_PEAK_F)
t_out = np.arange(WIND_PEAK_F, WIND_END_F)

# 入りの計算
wind_forces[WIND_START_F:WIND_PEAK_F] = WIND_MAX_STRENGTH * WIND_DIRECTION * np.sin((t_in - WIND_START_F) / (WIND_PEAK_F - WIND_START_F) * np.pi / 2)
# 終わりの計算
wind_forces[WIND_PEAK_F:WIND_END_F] = WIND_MAX_STRENGTH * WIND_DIRECTION * np.cos((t_out - WIND_PEAK_F) / (WIND_PEAK_F - WIND_END_F) * np.pi / 2)

print(f"🚀 {DURATION}秒ループ 生成開始... (Total: {total_frames} frames)")
start_time = time.time()

# フォント設定の固定化
font = cv2.FONT_HERSHEY_SIMPLEX
font_scale = 1.4
font_pos = (60, 120)

for f in range(total_frames):
    current_wind_force = wind_forces[f]

    # --- 爆発の実行 ---\
    if f in EXPLOSION_FRAMES:
        if f == 0:
            explosion_y = int(ROWS * PREFILL_HEIGHT * 0.5)
            velocity_grid, (ex, ey) = apply_deep_explosion(grid, velocity_grid)
            ey = explosion_y
        else:
            velocity_grid, (ex, ey) = apply_deep_explosion(grid, velocity_grid)

    # --- 砂の供給 ---\
    if f < EMIT_STOP_F:
        # ランダム生成をベクトル化
        sx_base = np.random.randint(COLS // 2 - 10, COLS // 2 + 11, 3)
        for sx in sx_base:
            if not np.any(grid[0, sx]):
                grid[0, sx] = get_dynamic_color()

    # 物理更新
    if abs(current_wind_force) > 0.001:
        velocity_grid = apply_wind(grid, velocity_grid, current_wind_force)

    grid, velocity_grid = update_physics_vectorized(grid, velocity_grid, MODE)

    # --- レンダリング (砂のみ回転) ---\
    # 最近傍補間は高速なのでそのまま維持
    sand_layer = cv2.resize(grid, (WIDTH, HEIGHT), interpolation=cv2.INTER_NEAREST)

    if f >= ROTATION_START_F:
        p = (f - ROTATION_START_F) / (total_frames - ROTATION_START_F)
        p_smooth = p * p * (3 - 2 * p)
        angle = p_smooth * 180
        center = (WIDTH // 2, HEIGHT // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        sand_layer = cv2.warpAffine(sand_layer, M, (WIDTH, HEIGHT), flags=cv2.INTER_NEAREST)

    # 背景コピーと合成
    frame = bg_img.copy()
    mask = np.any(sand_layer > 0, axis=2)
    frame[mask] = sand_layer[mask]

    # --- カウントダウンテキスト ---\
    if f < ROTATION_START_F:
        # 次のイベントまでの時間を計算
        next_events = [ef for ef in EXPLOSION_FRAMES if ef > f]
        if next_events:
            remaining = (next_events[0] - f) / FPS
            text = f"NEXT BOOM: {remaining:.1f}s"
            # cv2.LINE_AA は重いので LINE_4 に変更して少し軽量化
            cv2.putText(frame, text, font_pos, font, font_scale, (0, 0, 0), 10, cv2.LINE_4)
            cv2.putText(frame, text, font_pos, font, font_scale, (255, 255, 255), 3, cv2.LINE_4)

    frames_out.append(frame)

    # --- 進捗表示（5%刻みに変更してログ出力を削減） ---\
    if f % max(1, (total_frames // 20)) == 0:
        print(f"Progress: {f*100//total_frames}%")

print(f"\n✅ 完了！ ({DURATION}s) 処理時間: {time.time() - start_time:.2f}秒")

🚀 30秒ループ 生成開始... (Total: 900 frames)
Progress: 0%
Progress: 5%
Progress: 10%
Progress: 15%
Progress: 20%
Progress: 25%
Progress: 30%
Progress: 35%
Progress: 40%
Progress: 45%
Progress: 50%
Progress: 55%
Progress: 60%
Progress: 65%
Progress: 70%
Progress: 75%
Progress: 80%
Progress: 85%
Progress: 90%
Progress: 95%

✅ 完了！ (30s) 処理時間: 75.22秒


In [ ]:

import os
import time

# 保存パス（現在の設定を維持）
output_dir = "/content/drive/MyDrive/Colab Notebooks/output"
os.makedirs(output_dir, exist_ok=True)
temp_path = "/content/temp_raw.mp4"

timestamp = time.strftime("%Y%m%d_%H%M%S")
final_path = os.path.join(output_dir, f"sand_art_{DURATION}s_{timestamp}.mp4")

# 1. 一時ファイル生成
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_path, fourcc, FPS, (WIDTH, HEIGHT))

print("📦 映像データを生成中...")
for frame in frames_out:
    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
out.release()

# 2. ffmpegで爆速圧縮
# -preset ultrafast を追加して処理時間を大幅短縮
print("⚡ 爆速エンコード中（ultrafast設定）...")
!ffmpeg -y -i "{temp_path}" -vcodec libx264 -crf 23 -preset ultrafast -pix_fmt yuv420p -movflags +faststart "{final_path}" -loglevel error

# 一時ファイルの削除
if os.path.exists(temp_path):
    os.remove(temp_path)

# サイズ確認
file_size_mb = os.path.getsize(final_path) / (1024 * 1024)
print(f"\n✅ 完了！")
print(f"📂 保存先: {final_path}")
print(f"📦 サイズ: {file_size_mb:.2f} MB")

📦 映像データを生成中...
⚡ 爆速エンコード中（ultrafast設定）...

✅ 完了！
📂 保存先: /content/drive/MyDrive/Colab Notebooks/output/sand_art_30s_20260211_164115.mp4
📦 サイズ: 26.35 MB
